## Lesson 2: Connecting with a CRM

In [1]:
# Before you start, please run the following code to set up your environment.
# This code will reset the environment (if needed) and prepare the resources for the lesson.
# It does this by quickly running through all the code from the previous lessons.

!sh ./ro_shared_data/reset.sh
%run ./ro_shared_data/lesson_2_prep.py lesson2

import os

agentId = os.environ['BEDROCK_AGENT_ID']
agentAliasId = os.environ['BEDROCK_AGENT_ALIAS_ID']
region_name = 'us-west-2'
lambda_function_arn = os.environ['LAMBDA_FUNCTION_ARN']

Resetting environment (if nessesary)
Found: mugs-customer-support-agent
Deleting alias: MyAgentAlias (ID: N2YQNYIZYJ)
Deletion initiated for alias: MyAgentAlias
Alias MyAgentAlias has been successfully deleted.
Deleting alias: AgentTestAlias (ID: TSTALIASID)
Error deleting alias AgentTestAlias: An error occurred (ValidationException) when calling the DeleteAgentAlias operation: 1 validation error detected: Value 'TSTALIASID' at 'agentAliasId' failed to satisfy constraint: Member must satisfy regular expression pattern: (?!\bTSTALIASID\b)[0-9a-zA-Z]+
Deleting agent: mugs-customer-support-agent (ID: 8SOPTO6T5R)
Deletion initiated for agent: mugs-customer-support-agent
Waiting for agent mugs-customer-support-agent to be deleted...
Waiting for agent mugs-customer-support-agent to be deleted...
Agent mugs-customer-support-agent has been successfully deleted.
Agent reset process completed.
Lambda reset process completed.
Guardrail reset process completed.
Environment reset complete.
Lesson 2

## Start of lesson

In [2]:
import boto3
import uuid
from helper import *

In [3]:
sessionId = str(uuid.uuid4())
message = "My name is Mike, my mug is broken and I want a refund."

In [4]:
invoke_agent_and_print(
    agentId=agentId, 
    agentAliasId=agentAliasId, 
    inputText=message, 
    sessionId=sessionId
)

User: My name is Mike, my mug is broken and I want a refund.

Agent: I apologize, but I do not actually have access to any specific tools
       or functions to process customer refund requests. As a customer
       service agent, my role is to gather information from the
       customer and route the request to the appropriate team for
       further handling.  To properly assist Mike with his request for
       a refund on his broken mug, I will need to ask him some
       clarifying questions to understand the details of his purchase
       and the issue with the mug. This will allow me to properly
       document the request and escalate it to the correct team for
       review and processing.  <questions> - Mike, can you please
       provide the details of when and where you purchased the mug
       that is now broken? - Can you describe the nature of the damage
       or defect with the mug? - Do you have any order or transaction
       details, such as an order number or receip

<p style="background-color:#fff6ff; padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; The file that is examined in the video is at  <code>./ro_shared_data/functions/lambda_stage_1.py</code> </p>

In [5]:
bedrock_agent = boto3.client(service_name = 'bedrock-agent', region_name = region_name)

In [6]:
create_agent_action_group_response = bedrock_agent.create_agent_action_group(
    actionGroupName='customer-support-actions',
    agentId=agentId,
    actionGroupExecutor={
        'lambda': lambda_function_arn
    },
    functionSchema={
        'functions': [
            {
                'name': 'customerId',
                'description': 'Get a customer ID given available details. At least one parameter must be sent to the function. This is private information and must not be given to the user.',
                'parameters': {
                    'email': {
                        'description': 'Email address',
                        'required': False,
                        'type': 'string'
                    },
                    'name': {
                        'description': 'Customer name',
                        'required': False,
                        'type': 'string'
                    },
                    'phone': {
                        'description': 'Phone number',
                        'required': False,
                        'type': 'string'
                    },
                }
            },            
            {
                'name': 'sendToSupport',
                'description': 'Send a message to the support team, used for service escalation. ',
                'parameters': {
                    'custId': {
                        'description': 'customer ID',
                        'required': True,
                        'type': 'string'
                    },
                    'supportSummary': {
                        'description': 'Summary of the support request',
                        'required': True,
                        'type': 'string'
                    }
                }
            }
        ]
    },
    agentVersion='DRAFT',
)

In [7]:
create_agent_action_group_response

{'ResponseMetadata': {'RequestId': '4251dba9-c140-4b4b-aa2b-cbc05205acb4',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 10 Aug 2025 21:01:17 GMT',
   'content-type': 'application/json',
   'content-length': '1187',
   'connection': 'keep-alive',
   'x-amzn-requestid': '4251dba9-c140-4b4b-aa2b-cbc05205acb4',
   'x-amz-apigw-id': 'PG5ImF_OPHcEdCA=',
   'x-amzn-trace-id': 'Root=1-6899089d-3501fe61428843d579e506eb'},
  'RetryAttempts': 0},
 'agentActionGroup': {'actionGroupExecutor': {'lambda': 'arn:aws:lambda:us-west-2:092413168457:function:dlai-support-agent-MA774'},
  'actionGroupId': 'LGZB7EWGQW',
  'actionGroupName': 'customer-support-actions',
  'actionGroupState': 'ENABLED',
  'agentId': 'LOA9RKT42R',
  'agentVersion': 'DRAFT',
  'createdAt': datetime.datetime(2025, 8, 10, 21, 1, 17, 96036, tzinfo=tzlocal()),
  'functionSchema': {'functions': [{'description': 'Get a customer ID given available details. At least one parameter must be sent to the function. This is private 

In [8]:
actionGroupId = create_agent_action_group_response['agentActionGroup']['actionGroupId']


In [9]:
wait_for_action_group_status(
    agentId=agentId, 
    actionGroupId=actionGroupId,
    targetStatus='ENABLED'
)

Action Group status: ENABLED


'ENABLED'

In [10]:
bedrock_agent.prepare_agent(
    agentId=agentId
)

wait_for_agent_status(
    agentId=agentId,
    targetStatus='PREPARED'
)

Waiting for agent status of 'PREPARED'...
Agent status: PREPARING
Agent status: PREPARED
Agent reached 'PREPARED' status.


In [11]:
bedrock_agent.update_agent_alias(
    agentId=agentId,
    agentAliasId=agentAliasId,
    agentAliasName='MyAgentAlias',
)

wait_for_agent_alias_status(
    agentId=agentId,
    agentAliasId=agentAliasId,
    targetStatus='PREPARED'
)

Waiting for agent alias status of 'PREPARED'...
Agent alias status: UPDATING
Agent alias status: UPDATING
Agent alias status: PREPARED
Agent alias reached status 'PREPARED'


### Now use the agent with functions

In [12]:
sessionId = str(uuid.uuid4())
message = "My name is Mike (mike@mike.com), my mug is broken and I want a refund."

In [13]:
invoke_agent_and_print(
    agentId=agentId,
    agentAliasId=agentAliasId,
    inputText=message,
    sessionId=sessionId,
    enableTrace=False
)

User: My name is Mike (mike@mike.com), my mug is broken and I want a refund.

Agent: I apologize, but I'm unable to fully process your refund request for
       the broken mug at this time. It appears I'm missing some key
       details that are required to escalate this to our support team.
       Could you please provide any additional information about your
       order or the issue with the mug? I'd be happy to route this to
       the appropriate team once I have all the necessary details.
       Thank you for your patience.

Session ID: 466b968f-65e6-4f13-bd6a-149d9731a676


In [14]:
invoke_agent_and_print(
    agentId=agentId,
    agentAliasId=agentAliasId,
    inputText=message,
    sessionId=sessionId,
    enableTrace=True
)

User: My name is Mike (mike@mike.com), my mug is broken and I want a refund.

Agent: 
Agent's thought process:
  Okay, let's try processing this request again step-by-step:  1. The
  customer has provided their name (Mike) and email address
  (mike@mike.com), indicating they are an existing customer. 2. The
  customer is reporting a broken mug and requesting a refund. 3. I
  will first need to look up the customer's account details to verify
  their identity and order history before processing the refund
  request.

Invocation Input:
  Type: ACTION_GROUP
  Action Group: customer-support-actions
  Function: customerId
  Parameters: [{'name': 'name', 'type': 'string', 'value': 'Mike'}, {'name': 'email', 'type': 'string', 'value': 'mike@mike.com'}]

Observation:
  Type: ACTION_GROUP
  Action Group Output: {'id':5537}

Agent's thought process:
  Okay, I was able to look up the customer's account details using the
  name and email provided. The customer ID is 5537.  Now that I have
  the cu